In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:

from pyspark_data_provenance import (
    data_provenance_enabled, 
    build_data_provenance_session
)

In [15]:
from pyspark.sql import SparkSession

spark = (
    # SparkSession
    # .builder
    build_data_provenance_session()
    .appName("data-provenance-notebook")
    .getOrCreate()
)

In [16]:
print(spark.conf.get("spark.provenance.enabled", "false"))

with data_provenance_enabled(spark):
    print(spark.conf.get("spark.provenance.enabled", "false"))

print(spark.conf.get("spark.provenance.enabled", "false"))

false
true
false


In [17]:
from datetime import date

df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("C", date(2026, 1, 17), 80.0, 60),
    ("F", date(2026, 1, 16), 50.0, 70)
], ["product", "date", "price", "sales"]
)
df.printSchema()

df.toPandas()

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales: long (nullable = true)



,product,date,price,sales
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,C,2026-01-17,80.0,60
6,F,2026-01-16,50.0,70


In [18]:
df2 = spark.createDataFrame([
    ("A", "Bike"),
    ("B", "Handball"),
    ("C", "Bike"),
    ("D", "Handball"),
    ("E", "Running")
],["product","name"]
)

df2.printSchema()
df2.toPandas()



root
 |-- product: string (nullable = true)
 |-- name: string (nullable = true)



,product,name
0,A,Bike
1,B,Handball
2,C,Bike
3,D,Handball
4,E,Running


## Tests for WHERE

### SQL

In [19]:
df.createOrReplaceTempView("sales")
spark.sql("select * from sales where price>10").toPandas()

,product,date,price,sales
0,B,2026-01-15,100.0,20
1,B,2026-01-16,100.0,30
2,C,2026-01-17,80.0,60
3,F,2026-01-16,50.0,70


### Spark

In [20]:
df3 = df.select("*").filter("price > 10")
df3.toPandas()
#df3.explain("extended")

,product,date,price,sales
0,B,2026-01-15,100.0,20
1,B,2026-01-16,100.0,30
2,C,2026-01-17,80.0,60
3,F,2026-01-16,50.0,70


In [21]:
#df.createOrReplaceTempView("sales")

#with data_provenance_enabled(spark):
    #res = spark.sql("select * from sales where price>10").toPandas()

#res

In [22]:
df4 = df.select("*").filter("price > 10").sort("price")
df4.toPandas()

,product,date,price,sales
0,F,2026-01-16,50.0,70
1,C,2026-01-17,80.0,60
2,B,2026-01-15,100.0,20
3,B,2026-01-16,100.0,30


In [23]:
df5 = df.select("*").join(df2,"product")
df5.toPandas()

,product,date,price,sales,name
0,A,2026-01-15,10.0,90,Bike
1,A,2026-01-16,10.0,120,Bike
2,A,2026-01-17,5.0,300,Bike
3,B,2026-01-15,100.0,20,Handball
4,B,2026-01-16,100.0,30,Handball
5,C,2026-01-17,80.0,60,Bike


In [24]:
df6 = df.select("*").join(df2, "product", "outer")
df6.toPandas()


,product,date,price,sales,name
0,A,2026-01-15,10.0,90.0,Bike
1,A,2026-01-16,10.0,120.0,Bike
2,A,2026-01-17,5.0,300.0,Bike
3,B,2026-01-15,100.0,20.0,Handball
4,B,2026-01-16,100.0,30.0,Handball
5,C,2026-01-17,80.0,60.0,Bike
6,D,None,NaN,NaN,Handball
7,E,None,NaN,NaN,Running
8,F,2026-01-16,50.0,70.0,NaN


In [37]:
df7 = df.groupBy("product").agg({"sales": "sum"}).sort("product")
df7.toPandas()

,product,sum(sales)
0,A,510
1,B,50
2,C,60
3,F,70


In [43]:
df8 = df.withColumn("revenue", df["sales"] * df["price"]) \
        .groupBy("product") \
        .agg({"revenue": "sum"}) \
        .sort("product")

df8.toPandas()


,product,sum(revenue)
0,A,3600.0
1,B,5000.0
2,C,4800.0
3,F,3500.0
